# Processamento do Dataset google/Synthetic-Persona-Chat

Este notebook processa o dataset `google/Synthetic-Persona-Chat` para:
1. Extrair todas as descrições únicas de persona
2. Gerar IDs determinísticos (SHA-256) baseados no conteúdo da descrição
3. Criar um **dataset enriquecido** com colunas `persona-1-id`, `persona-2-id` e `split`
4. Criar um **dataset de mapeamento** `persona-id → description`

In [2]:
import hashlib
from datasets import load_dataset, Dataset, concatenate_datasets


def generate_persona_id(description: str) -> str:
    """Gera um ID determinístico SHA-256 a partir da descrição da persona."""
    return hashlib.sha256(description.encode("utf-8")).hexdigest()

## 1. Carregar o dataset (todos os splits)

In [3]:
dataset_dict = load_dataset("google/Synthetic-Persona-Chat")

print(f"Total de linhas (todos os splits): {sum(len(ds) for ds in dataset_dict.values())}")
for split_name, ds in dataset_dict.items():
    print(f"  {split_name}: {len(ds)} linhas")

Total de linhas (todos os splits): 10906
  train: 8938 linhas
  validation: 1000 linhas
  test: 968 linhas


## 2. Coletar descrições únicas de persona (todos os splits)

In [4]:
unique_personas: set[str] = set()
for ds in dataset_dict.values():
    for row in ds:
        for col in ("user 1 personas", "user 2 personas"):
            desc = row[col].strip()
            if desc:
                unique_personas.add(desc)

print(f"Descrições únicas encontradas: {len(unique_personas)}")

Descrições únicas encontradas: 21059


## 3. Gerar IDs determinísticos (SHA-256)

In [5]:
desc_to_id: dict[str, str] = {}
for desc in unique_personas:
    desc_to_id[desc] = generate_persona_id(desc)

print(f"IDs gerados: {len(desc_to_id)}")
print(f"Exemplo de ID: {list(desc_to_id.values())[0]}")

IDs gerados: 21059
Exemplo de ID: 37825969ae2d5329e71fdb030a2d8e740b66257af1e8491d742b680117571746


## 4. Criar dataset enriquecido (com IDs e coluna split)

In [6]:
from datasets import DatasetDict

enriched_dataset_dict = DatasetDict()
for split_name, ds in dataset_dict.items():
    enriched_data = []
    for row in ds:
        new_row = dict(row)
        new_row["persona-1-id"] = desc_to_id[row["user 1 personas"].strip()]
        new_row["persona-2-id"] = desc_to_id[row["user 2 personas"].strip()]
        enriched_data.append(new_row)
    enriched_dataset_dict[split_name] = Dataset.from_list(enriched_data)

print("Dataset enriquecido (estrutura DatasetDict):")
for split_name, ds in enriched_dataset_dict.items():
    print(f"  {split_name}: {len(ds)} linhas")
print(f"Colunas: {enriched_dataset_dict['train'].column_names}")

Dataset enriquecido (estrutura DatasetDict):
  train: 8938 linhas
  validation: 1000 linhas
  test: 968 linhas
Colunas: ['user 1 personas', 'user 2 personas', 'Best Generated Conversation', 'persona-1-id', 'persona-2-id']


## 5. Criar dataset de mapeamento (persona-id → description)

In [7]:
mapping_data = [
    {"persona-id": pid, "description": desc}
    for desc, pid in desc_to_id.items()
]
mapping_dataset = Dataset.from_list(mapping_data)

print(f"Dataset de mapeamento: {len(mapping_dataset)} personas únicas")
print(f"Colunas: {mapping_dataset.column_names}")

Dataset de mapeamento: 21059 personas únicas
Colunas: ['persona-id', 'description']


## 6. Publicar no HuggingFace Hub

In [1]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
enriched_dataset_dict.push_to_hub("visual-memory/Synthetic-Persona-Chat-With-Ids")
mapping_dataset.push_to_hub("visual-memory/Synthetic-Persona-Chat-Mapping")

print("Dataset enriquecido publicado!")
print("Dataset de mapeamento publicado!")

sample = enriched_dataset_dict["train"][0]
print(f"\nExemplo de linha enriquecida (split 'train'):")
print(f"  persona-1-id: {sample['persona-1-id']}")
print(f"  persona-2-id: {sample['persona-2-id']}")

Setting num_proc from 1 back to 1 for the train split to disable multiprocessing as it only contains one shard.


Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Dataset enriquecido publicado!
Dataset de mapeamento publicado!

Exemplo de linha enriquecida (split 'train'):
  persona-1-id: 2e2f3da53072d0b29bda1901902a201f26e2ceaa79695336be1d24de2a4e65fb
  persona-2-id: 10a2753eea8aeca83aa5f407217ef08cacddc56e67307daa64caa7b05cb8cd2b
